# LangChain agents
In this part we practice with LangChain agent that can create and test code for us.

In [ ]:
from langchain.agents import create_agent
from langchain_experimental.tools import PythonREPLTool 
from langchain_community.document_loaders import WebBaseLoader
from langchain_openai import ChatOpenAI

from IPython.display import Markdown, display


LangChain manages the system prompting and the execution of tools. We only need to provide it an LLM it can use in the background

In [ ]:
api_key = ""   # Replace with your actual OpenAI API key

llm = ChatOpenAI(api_key=api_key, model="gpt-5.4-mini-2026-03-17")  # or model="gpt-4"

#### Example 1, A webscrape tool:
The agent is able to acces certain (public) websites and retrieve information from it as context when answering questions. Below is an example
- Test the script and see that it uses nos.nl as context
- Try a couple of other websites and see if it can give quick summaries for you


In [ ]:
def scrape_url(url: str) -> str:
    """Scrape the text content of a webpage."""
    docs = WebBaseLoader(url).load()
    return docs[0].page_content[:5000]

agent = create_agent(llm, [scrape_url])

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Summarise the most recent news events from: https://nos.nl"
    }]
})

print(result["messages"][-1].content)

#### Example 2, The Python REPL tool
We can also the agent with a REPL (Read-Eval-Print Loop) tool, that can exectue and evaluate code and assist in more detailed debugging (based on output of the code). As AI can execute code there is risk of running harmful code (i.e. code that deletes files). Therefore one can also use the safe_python_repl tool, which always asks for permission before executing code. 

In [ ]:
# Use either the basic REPL tool or a safer version (human in the loop)

base_repl = PythonREPLTool()

def safe_python_repl(code: str) -> str:
    """
    Use this tool to write and execute Python code.
    Use it for calculations, simulations, data analysis, or anything that requires running code.
    Input should be valid Python code as a string.
    """
    print("\n--- CODE TO BE EXECUTED ---")
    print(code)
    print("---------------------------")
    approval = input("Approve execution? (yes/no): ").strip().lower()
    if approval == "yes":
        return base_repl.run(code)
    return "Execution rejected by user."

In [ ]:
tools = [PythonREPLTool()]
# tools = [safe_python_repl()]

agent = create_agent(llm, tools)

#### Example 1, Generate code with the REPL tool for data analysis
Below we provide an example of how an agent can read in data and perform basic data analysis on it
- Read the system prompt and user prompt and run the script
- Try to ask it additional things, for example for it to make different plots

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "system",
            "content": """You are a data science assistant with access to a Python REPL.
                            When asked to build a model, always:
                            1. Load and inspect the data first
                            2. Clean and prepare it
                            3. Build the model
                            4. Print clear results and metrics
                            5. Save any plots as PNG files

                            The file stock_prices.xlsx is in the current working directory.
                            Always use the columns 'Date' and 'Close'.
                            """
        },
        {
            "role": "user",
            "content": """
            Load stock_prices.xlsx and build a price prediction model using linear regression based on prices of the last few months"
            lots actual vs predicted prices
            """
        }
    ]
})

print(result["messages"][-1].content)

#### Example 2, Physics code

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "system",
            "content": """You are a Python coding assistant with access to a Python REPL.
                        When asked to simulate or compute something, always:
                        1. Write a clean, well-commented Python function
                        2. Execute it to verify correctness
                        3. Print the results in a clear, readable format
                        4. Show the code used

                        Always print both the code and its output explicitly.
                        """
        },
        {
            "role": "user",
            "content": """
            Simulate the motion of a ball dropped from 100 meters height under gravity (g = 9.81 m/s^2).
            Write a Python function that computes height after time t.
            Then calculate the height at t=0, 1, 2, 3, 4, and 5 seconds.
            Additionally provide the code used.
            """
        }
    ]
})

print(result["messages"][-1].content)

#### Example 3, Testing code
We can also use the REPL tool for an agent to verify code for us

In [11]:
agent = create_agent(
    llm,
    tools,
    system_prompt=(
        "You are a Python code reviewer and fixer. "
        "When given buggy code and a set of business rules:\n"
        "1. Run the code AS-IS and observe all failures.\n"
        "2. For each failure, explain in plain English what went wrong and why.\n"
        "3. Fix every bug.\n"
        "4. Re-run and confirm ALL rules pass.\n"
        "5. Return a final summary containing:\n"
        "   - A bullet list of every bug found, with a plain-English explanation of the mistake\n"
        "   - The corrected code in a clearly marked block.\n"
        "Never ask the user for expected outputs — derive correctness from the rules alone.\n"
        "Write your explanations so a non-technical person can understand what was broken."
    ),
)

In [12]:
task = f"""

--- BUGGY CODE ---
def calculate_discount(price, discount_pct):
    
    discount = price * (discount_pct / 100)
    return price - discount + 1  
--- END ---
"""

We added a custom print cell, so that you can see which steps the agent makes

In [ ]:

response = agent.invoke({"messages": [{"role": "user", "content": task}]})

for msg in response["messages"]:
    role = getattr(msg, "type", type(msg).__name__)
    content = msg.content if isinstance(msg.content, str) else str(msg.content)
    if content.strip():
        print(f"\n{'─' * 60}")
        print(f"[{role.upper()}]")
        print(content)


────────────────────────────────────────────────────────────
[HUMAN]


--- BUGGY CODE ---
def calculate_discount(price, discount_pct):

    discount = price * (discount_pct / 100)
    return price - discount + 1  
--- END ---


────────────────────────────────────────────────────────────
[TOOL]
(100, 10) 91.0
(200, 0) 201.0
(50, 25) 38.5


────────────────────────────────────────────────────────────
[TOOL]
(100, 10) 90.0
(200, 0) 200.0
(50, 25) 37.5


────────────────────────────────────────────────────────────
[AI]
Here’s what was broken and how it was fixed:

- **The function added an extra `+ 1` to the final price.**
  - Plain English: after calculating the discount, it was making the result $1 too high every time.
  - Why that’s wrong: a discount should only subtract the discounted amount from the original price, not add anything extra.

### Corrected code
```python
def calculate_discount(price, discount_pct):
    discount = price * (discount_pct / 100)
    return price - discount